# Optimising a site with a load and three evs - one v0g, one v1g, and one v2g

## Import required modules

In [33]:
import numpy as np
from echo.echo_models import *
from echo.echo_optimiser import *
from echo.objectives import *

## Define model parameters


In [34]:
time_periods = 96  # total number of intervals
interval_duration = 15  # Duration in mins of each interval
expansion_periods = 1  # Number of planning intervals - in echo V1, set to 1 always

## Define data as arrays
These can either be defined as lists or numpy arrays

In [35]:
load_array = np.array(
    [2.13, 2.09, 2.3, 2.11, 2.2, 2.23, 15, 15, 15, 2.19, 2.19, 2.19, 2.12, 2.15, 2.25, 2.12, 2.21, 2.16,
     2.26, 2.13, 2.08, 2.15, 2.42, 2.02, 2.3, 2.26, 2.35, 2.55, 3.23, 2.98, 3.49, 3.5, 3.12, 3.52, 3.94, 3.55,
     3.99, 3.71, 3.38, 3.76, 3.71, 3.78, 3.29, 3.65, 3.61, 3.75, 3.38, 3.66, 3.56, 3.69, 3.3, 3.61, 3.71, 3.82,
     3.17, 3.69, 3.74, 3.86, 3.57, 3.55, 3.75, 3.6, 3.67, 3.48, 3.51, 3.46, 3.19, 3.38, 3.19, 3.38, 3.04, 3.12,
     2.91, 3.11, 3.13, 2.77, 2.24, 2.54, 2.24, 2.24, 2.09, 2.33, 2.17, 2.16, 1.97, 2.16, 2.21, 2.18, 2.01, 2.16,
     2.19, 2.11, 2.17, 2.13, 12, 12])

## Instantiate an echo optimisation graph

In [36]:
system = OptimisationGraph()

### Create grid node

In [37]:
grid = Node() # create node representing upstream grid
grid.add_named_electrical_ports(['grid'])

### Create connection point node

In [38]:
connection_point = ElectricalTellegenNode()     # create the connection point node
connection_point.add_named_electrical_ports(['grid', 'load', 'ev_v0g', 'ev_v1g', 'ev_v2g'])  # add ports with easily referenced names

### Create load node
Load nodes typically have a single port.

In [39]:
load = Node()                       # create a node to represent the load
load_port = ElectricalDemand()             # create an electrical demand port to attach to this node
load_port.add_demand_profile_from_array(load_array, expansion_periods)
load.ports['load'] = load_port # explicitly add the load_port to the dictionary of node ports

## Create EV nodes
### V0G
V0G convenience charges (charges at max capacity until full) when it is plugged in.

In [40]:
v0g_availability = [1]*48 + [0]*48  # binary array, 1 indicates it is plugged in and can charge
v0g_trip_usage = [0]*48 + [0.5]*48  # in average kW per interval

v0g = EV(charge_mode='V0G',
               available=v0g_availability,
               usage=v0g_trip_usage,
               connection_port_name='cp',
               max_capacity=40,
               depth_of_discharge_limit=0,
               charging_power_limit=10,
               discharging_power_limit=-1e4,
               charging_efficiency=1,
               discharging_efficiency=1,
               initial_state_of_charge=20,
               soc_conserv=None,
               soc_conserv_cost=0.,
               interval_duration=interval_duration,
               tod_charging=None,
               trip_slack=True)

ValidationError: 1 validation error for EV
tod_charging
  none is not an allowed value (type=type_error.none.not_allowed)

In [ ]:
# create V1G vehicle

v1g_availability = [1]*48 + [0]*48  # binary array, 1 indicates it is plugged in and can charge
v1g_trip_usage = [0]*48 + [2]*48  # in average kW per interval

v1g = EV(charge_mode='V1G',
         available=v1g_availability,
         usage=v1g_trip_usage,
         connection_port_name='cp',
         max_capacity=40,
         depth_of_discharge_limit=0.,
         charging_power_limit=10.,
         discharging_power_limit=-10.,
         charging_efficiency=1.,
         discharging_efficiency=1.,
         initial_state_of_charge=0.,
         tod_charging=None,
         soc_conserv=None,
         soc_conserv_cost=0.,
         trip_slack=True,
         interval_duration=interval_duration)


In [ ]:
# create V2G vehicle

v2g_availability = [1]*48 + [0]*48  # binary array, 1 indicates it is plugged in and can charge
v2g_trip_usage = [0]*48 + [5]*48  # in average kW per interval

v2g = EV(charge_mode='V2G',
         available=v2g_availability,
         usage=v2g_trip_usage,
         connection_port_name='cp',
         max_capacity=40,
         depth_of_discharge_limit=0.,
         charging_power_limit=10.,
         discharging_power_limit=-10.,
         charging_efficiency=1.,
         discharging_efficiency=1.,
         initial_state_of_charge=0.,
         tod_charging=None,
         soc_conserv=None,
         soc_conserv_cost=0.,
         trip_slack=True,
         interval_duration=interval_duration)

### Add all nodes to the graph

In [ ]:
system.add_node_obj([grid, connection_point, load, v0g, v1g, v2g])  # nodes can be added one by one or as a list

### Make connections between ports on nodes
This does two things - it creates an edge object for each edge, and it adds that edge object to the optimisation graph.

In [ ]:
# Add edges to graph
system.connect_ports_and_create_edge(grid.ports['grid'], connection_point.ports['grid'])
system.connect_ports_and_create_edge(connection_point.ports['load'], load.ports['load'])
system.connect_ports_and_create_edge(connection_point.ports['ev_v0g'], v0g.ports['cp'])
system.connect_ports_and_create_edge(connection_point.ports['ev_v1g'], v1g.ports['cp'])
system.connect_ports_and_create_edge(connection_point.ports['ev_v2g'], v2g.ports['cp'])

## Create objectives
Here we will create an import tariff and export tariff. These tariffs are echo objects.

## Define tariff arrays

In [ ]:
# Tariffs are in $ / kWh
import_tariff_array = np.array(([0.1] * 28 + [0.3] * 8 + [0.2] * 32 + [0.3] * 16 + [0.1] * 12))
export_tariff_array = np.array(([0.0] * 96))

### Create echo objective set

In [ ]:
# First create the tariffs as echo objects
import_cost = ImportTariff(component=connection_point.ports['grid'],
                           tariff_array=import_tariff_array,
                           expansion_periods=expansion_periods)  # create the import objective cost

export_cost = ExportTariff(component=connection_point.ports['grid'],
                           tariff_array=export_tariff_array,
                           expansion_periods=expansion_periods)  # create the export objective cost

# Then define the objective set, which is just a list with the two costs
objective_set = ObjectiveSet(objective_list=[import_cost, export_cost])


## Run the optimiser

In [ ]:
# Invoke the optimiser
optimiser = EchoOptimiser(interval_duration=interval_duration,
                          number_of_intervals=time_periods,
                          number_of_expansion_intervals=expansion_periods,
                          discount_rate=0,
                          ES=system,
                          objective_set=objective_set,
                          optimiser_engine='cplex')

# Optimise
optimiser.optimise()

## Plot some results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get optimised results
# These are retrieved by calling the optimiser .values() method, and providing the variable name we want.
# In this case, the storage port has two variables of interest - port_name, which is the battery power, and soc_value

v0g_delta = optimiser.values(v0g.ports['vehicle'].port_name, 0)
v0g_soc = optimiser.values(v0g.ports['vehicle'].soc_value, 0)

v1g_delta = optimiser.values(v1g.ports['vehicle'].port_name, 0)
v1g_soc = optimiser.values(v1g.ports['vehicle'].soc_value, 0)

v2g_delta = optimiser.values(v2g.ports['vehicle'].port_name, 0)
v2g_soc = optimiser.values(v2g.ports['vehicle'].soc_value, 0)

optimised_connection_point_load = optimiser.values(connection_point.ports['grid'].port_name, 0)

# Do plotting
colors = sns.color_palette()
hrs = np.arange(0, len(load_array)) / 4
fig = plt.figure(figsize=(14, 4))
ax1 = fig.add_subplot(1, 1, 1)
line1, = ax1.plot(hrs, load_array, color=colors[0])
line3, = ax1.plot(hrs, optimised_connection_point_load, color=colors[2])
ax1.set_xlabel('hour'), ax1.set_ylabel('kW')
ax1.legend([line1, line3], ['Load', 'connection pt'], ncol=2)
ax1.set_xlim([0, len(load_array) / 4])

fig = plt.figure(figsize=(14, 4))
ax2 = fig.add_subplot(1, 1, 1)
line1, = ax2.plot(hrs, import_tariff_array, color=colors[3])
line2, = ax2.plot(hrs, export_tariff_array, color=colors[4])
ax2.set_xlabel('hour'), ax2.set_ylabel('price')
ax2.legend([line1, line2], ['buy price', 'sell price'], ncol=2)
ax2.set_xlim([0, len(load_array) / 4])

fig = plt.figure(figsize=(14, 4))
ax3 = fig.add_subplot(1, 1, 1)
line1, = ax3.plot(hrs, v0g_delta, color=colors[1])
line2, = ax3.plot(hrs, v0g_soc, color=colors[2])
line3, = ax3.plot(hrs, v0g_trip_usage, color=colors[3])
ax3.set_xlim([0, len(load_array) / 4])
ax3.set_xlabel('hour')
ax3.legend([line1, line2, line3], ['V0G Charging action (kW)', 'SOC (kWh)', 'trip usage'])
plt.show()

fig = plt.figure(figsize=(14, 4))
ax4 = fig.add_subplot(1, 1, 1)
line1, = ax4.plot(hrs, v1g_delta, color=colors[1])
line2, = ax4.plot(hrs, v1g_soc, color=colors[2])
line3, = ax4.plot(hrs, v1g_trip_usage, color=colors[3])
ax4.set_xlim([0, len(load_array) / 4])
ax4.set_xlabel('hour')
ax4.legend([line1, line2, line3], ['V1G Charging action (kW)', 'SOC (kWh)', 'trip usage'])
plt.show()

fig = plt.figure(figsize=(14, 4))
ax5 = fig.add_subplot(1, 1, 1)
line1, = ax5.plot(hrs, v2g_delta, color=colors[1])
line2, = ax5.plot(hrs, v2g_soc, color=colors[2])
line3, = ax5.plot(hrs, v2g_trip_usage, color=colors[3])
ax5.set_xlim([0, len(load_array) / 4])
ax5.set_xlabel('hour')
ax5.legend([line1, line2, line3], ['V2G Charging action (kW)', 'SOC (kWh)', 'trip usage'])
plt.show()


### Print some data

In [ ]:
print(optimiser.values(v2g.ports['vehicle'].trip_slack, 0))
print(optimiser.values(v2g.ports['vehicle'].soc_value, 0))
print(optimiser.values(v2g.ports['cp'].port_name, 0))

